# Phase 1 Final Controls Remediation Audit

This notebook is orchestration only. It audits failed final controls after the claim-closed remediation plan.

Scientific integrity rules:

- This notebook does not rerun controls, retrain models, edit logits, edit metrics, or change thresholds.
- It reads the failed final controls and dedicated-control artifacts exactly as generated.
- It may classify failure reasons and threshold-source consistency for remediation review.
- It must keep `claims_opened = false` and must not reinterpret failed controls as pass.
- Any future code/config change must be handled as revision-scoped remediation, not as a post-hoc threshold edit.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Preferred: pin reviewed sources explicitly. Leave as None to resolve the latest valid run.
# Do not type placeholder run-id strings into these fields.
PINNED_FINAL_REMEDIATION_PLAN_RUN = None
PINNED_FINAL_CONTROLS_RUN = None
PINNED_FINAL_DEDICATED_CONTROLS_RUN = None

def latest_run_with_file(root: Path, required_file: str) -> Path:
    candidates = sorted([
        p for p in root.iterdir()
        if p.is_dir() and (p / required_file).exists()
    ])
    if not candidates:
        raise FileNotFoundError(f'No runs with {required_file} under {root}')
    print(f'Found runs under {root}:', len(candidates))
    for p in candidates[-5:]:
        print(p, 'required file exists =', (p / required_file).exists())
    return candidates[-1]

if PINNED_FINAL_REMEDIATION_PLAN_RUN is None:
    FINAL_REMEDIATION_PLAN_RUN = latest_run_with_file(
        ARTIFACT_ROOT / 'phase1_final_remediation_plan',
        'phase1_final_remediation_plan_summary.json',
    )
else:
    FINAL_REMEDIATION_PLAN_RUN = Path(PINNED_FINAL_REMEDIATION_PLAN_RUN)

if PINNED_FINAL_CONTROLS_RUN is None:
    FINAL_CONTROLS_RUN = latest_run_with_file(
        ARTIFACT_ROOT / 'phase1_final_controls',
        'final_control_manifest.json',
    )
else:
    FINAL_CONTROLS_RUN = Path(PINNED_FINAL_CONTROLS_RUN)

if PINNED_FINAL_DEDICATED_CONTROLS_RUN is None:
    FINAL_DEDICATED_CONTROLS_RUN = latest_run_with_file(
        ARTIFACT_ROOT / 'phase1_final_dedicated_controls',
        'final_dedicated_control_manifest.json',
    )
else:
    FINAL_DEDICATED_CONTROLS_RUN = Path(PINNED_FINAL_DEDICATED_CONTROLS_RUN)

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_remediation_audit'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_remediation_audit_plan'

RUN_FINAL_CONTROLS_REMEDIATION_AUDIT = False
REQUIRED_ACK = 'I reviewed final controls remediation audit and understand it records failed controls without changing thresholds or opening claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_controls_remediation_audit_v2_latest_sources_20260422'

print('Notebook fix marker:', FIX_MARKER)
print('Remediation plan run:', FINAL_REMEDIATION_PLAN_RUN)
print('Remediation summary exists:', (FINAL_REMEDIATION_PLAN_RUN / 'phase1_final_remediation_plan_summary.json').exists())
print('Final controls run:', FINAL_CONTROLS_RUN)
print('Final control manifest exists:', (FINAL_CONTROLS_RUN / 'final_control_manifest.json').exists())
print('Final dedicated controls run:', FINAL_DEDICATED_CONTROLS_RUN)
print('Final dedicated control manifest exists:', (FINAL_DEDICATED_CONTROLS_RUN / 'final_dedicated_control_manifest.json').exists())
print('Run flag:', RUN_FINAL_CONTROLS_REMEDIATION_AUDIT)


In [ ]:
# Cell 3 - Utilities and prereg/source validation.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '
', encoding='utf-8')

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_prereg_bundle(path: Path) -> Path:
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Found prereg bundles:', len(candidates))
    for candidate in candidates:
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_IDENTITY_HASH:
            return candidate
    raise FileNotFoundError(f'No locked prereg bundle found for expected identity hash under {ARTIFACT_ROOT / "prereg"}')

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
FINAL_REMEDIATION_PLAN_RUN = Path(FINAL_REMEDIATION_PLAN_RUN)
FINAL_CONTROLS_RUN = Path(FINAL_CONTROLS_RUN)
FINAL_DEDICATED_CONTROLS_RUN = Path(FINAL_DEDICATED_CONTROLS_RUN)

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

remediation_summary = read_json(FINAL_REMEDIATION_PLAN_RUN / 'phase1_final_remediation_plan_summary.json')
controls_summary = read_json(FINAL_CONTROLS_RUN / 'phase1_final_controls_summary.json')
controls_manifest = read_json(FINAL_CONTROLS_RUN / 'final_control_manifest.json')
dedicated_summary = read_json(FINAL_DEDICATED_CONTROLS_RUN / 'phase1_final_dedicated_controls_summary.json')
dedicated_manifest = read_json(FINAL_DEDICATED_CONTROLS_RUN / 'final_dedicated_control_manifest.json')

assert remediation_summary.get('claims_opened') is False
assert remediation_summary.get('final_claim_blocked') is True
assert 'controls' in remediation_summary.get('blocking_surfaces', []), 'Remediation plan must mark controls as blocking.'
assert controls_summary.get('claim_ready') is False
assert controls_manifest.get('control_suite_passed') is False, 'This audit expects failed final controls.'
assert dedicated_summary.get('claim_ready') is False
assert dedicated_manifest.get('dedicated_control_suite_passed') is False, 'This audit expects failed dedicated controls.'

preflight = subprocess.run(
    ['python', '-m', 'src.cli', 'phase1_final_controls_remediation_audit', '--help'],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)
runner_available = preflight.returncode == 0
print('Runner available:', runner_available)
assert runner_available, preflight.stderr


In [ ]:
# Cell 4 - Record manual-hold plan and exact CLI command.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / created_utc
cmd = [
    'python', '-m', 'src.cli', 'phase1_final_controls_remediation_audit',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-remediation-plan-run', str(FINAL_REMEDIATION_PLAN_RUN),
    '--final-controls-run', str(FINAL_CONTROLS_RUN),
    '--final-dedicated-controls-run', str(FINAL_DEDICATED_CONTROLS_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--audit-config', 'configs/phase1/final_controls_remediation_audit.json',
    '--final-controls-config', 'configs/phase1/final_controls.json',
    '--dedicated-controls-config', 'configs/phase1/final_dedicated_controls.json',
    '--control-suite-config', 'configs/controls/control_suite_spec.yaml',
    '--gate2-config', 'configs/gate2/synthetic_validation.json',
]
plan = {
    'status': 'phase1_final_controls_remediation_audit_manual_hold_recorded',
    'created_utc': created_utc,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_CONTROLS_REMEDIATION_AUDIT,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'final_remediation_plan_run': str(FINAL_REMEDIATION_PLAN_RUN),
    'final_controls_run': str(FINAL_CONTROLS_RUN),
    'final_dedicated_controls_run': str(FINAL_DEDICATED_CONTROLS_RUN),
    'final_controls_passed': controls_manifest.get('control_suite_passed'),
    'dedicated_controls_passed': dedicated_manifest.get('dedicated_control_suite_passed'),
    'failed_dedicated_controls': dedicated_manifest.get('failed_results', []),
    'command': cmd,
    'scientific_limit': 'This audit records observed controls blockers only; it does not repair controls or open claims.',
}
write_json(plan_dir / 'phase1_final_controls_remediation_audit_manual_hold.json', plan)
print('Plan source of truth:', plan_dir)
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual hold or launch.

if not RUN_FINAL_CONTROLS_REMEDIATION_AUDIT:
    hold = {
        'status': 'phase1_final_controls_remediation_audit_manual_hold',
        'plan_dir': str(plan_dir),
        'run_flag': RUN_FINAL_CONTROLS_REMEDIATION_AUDIT,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    }
    print(json.dumps(hold, indent=2))
    print('HELD: Controls remediation audit was not launched because manual flag is False.')
    print('NEXT: review the source artifacts and plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch controls remediation audit without explicit claim-closed acknowledgement.')
else:
    write_json(plan_dir / 'phase1_final_controls_remediation_audit_launch_review.json', {
        'status': 'phase1_final_controls_remediation_audit_launch_reviewed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'ack': MANUAL_LAUNCH_ACK,
        'claims_opened': False,
        'scientific_limit': 'Launch records an audit only; it must not alter failed controls evidence.',
    })
    run(cmd, cwd=REPO_DIR)
    print('Final controls remediation audit command completed. Review generated artifacts before any code/config remediation.')


In [ ]:
# Cell 6 - Inspect latest controls-remediation-audit output.

latest_txt = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest_txt.exists())
if latest_txt.exists():
    latest_run = Path(latest_txt.read_text(encoding='utf-8').strip())
else:
    runs = sorted([path for path in OUTPUT_ROOT.iterdir() if path.is_dir()]) if OUTPUT_ROOT.exists() else []
    latest_run = runs[-1] if runs else None

if latest_run is None:
    print('No controls-remediation-audit output yet; this is expected for manual hold.')
else:
    print('Latest final controls-remediation-audit output:', latest_run)
    expected_files = [
        'phase1_final_controls_remediation_audit_inputs.json',
        'phase1_final_controls_remediation_audit_summary.json',
        'phase1_final_controls_remediation_audit_report.md',
        'phase1_final_controls_remediation_source_links.json',
        'phase1_final_controls_remediation_input_validation.json',
        'phase1_final_controls_failure_table.json',
        'phase1_final_controls_threshold_source_review.json',
        'phase1_final_controls_implementation_review.json',
        'phase1_final_controls_remediation_work_items.json',
        'phase1_final_controls_remediation_claim_state.json',
        'phase1_final_controls_remediation_decision_memo.md',
    ]
    for name in expected_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_controls_remediation_audit_summary.json')
    print(json.dumps({
        'status': summary.get('status'),
        'claim_ready': summary.get('claim_ready'),
        'claims_opened': summary.get('claims_opened'),
        'control_suite_passed': summary.get('control_suite_passed'),
        'dedicated_control_suite_passed': summary.get('dedicated_control_suite_passed'),
        'failed_dedicated_controls': summary.get('failed_dedicated_controls'),
        'blocking_controls': summary.get('blocking_controls'),
        'teacher_threshold_path_mismatch_suspected': summary.get('teacher_threshold_path_mismatch_suspected'),
        'scientific_limit': summary.get('scientific_limit'),
    }, indent=2))


In [ ]:
# Cell 7 - Assertions and local review note.

if RUN_FINAL_CONTROLS_REMEDIATION_AUDIT:
    assert latest_run is not None, 'Expected controls-remediation-audit output after launch.'
    summary = read_json(latest_run / 'phase1_final_controls_remediation_audit_summary.json')
    failure_table = read_json(latest_run / 'phase1_final_controls_failure_table.json')
    threshold_review = read_json(latest_run / 'phase1_final_controls_threshold_source_review.json')
    implementation_review = read_json(latest_run / 'phase1_final_controls_implementation_review.json')
    claim_state = read_json(latest_run / 'phase1_final_controls_remediation_claim_state.json')

    expected_dedicated_controls = {
        'nuisance_shared_control',
        'spatial_control',
        'shuffled_teacher',
        'time_shifted_teacher',
    }
    dedicated_rows = {
        row.get('control_id'): row
        for row in failure_table.get('rows', [])
        if row.get('control_type') == 'dedicated'
    }
    observed_failed_dedicated = set(summary.get('failed_dedicated_controls', []))
    table_failed_dedicated = set(failure_table.get('failed_dedicated_controls', []))
    observed_blocking_controls = set(summary.get('blocking_controls', []))
    table_blocking_controls = set(failure_table.get('blocking_controls', []))

    assert summary.get('status') == 'phase1_final_controls_remediation_audit_recorded', summary
    assert summary.get('claim_ready') is False
    assert summary.get('claims_opened') is False
    assert summary.get('final_claim_blocked') is True
    assert summary.get('control_suite_passed') is False
    assert summary.get('dedicated_control_suite_passed') is False
    assert expected_dedicated_controls <= set(dedicated_rows), 'All required dedicated-control rows must be audited.'
    assert observed_failed_dedicated == table_failed_dedicated, 'Summary and failure table must agree on observed failed dedicated controls.'
    assert observed_failed_dedicated <= expected_dedicated_controls, 'Unexpected dedicated-control id in failed list.'
    assert observed_blocking_controls == table_blocking_controls, 'Summary and failure table must agree on blocking controls.'
    assert observed_blocking_controls, 'Expected observed blocking controls to remain recorded.'
    assert claim_state.get('headline_phase1_claim_open') is False

    review_note = {
        'status': 'phase1_final_controls_remediation_audit_review_pass_claim_closed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_run),
        'checks_passed': [
            'required_artifacts_present',
            'claim_ready_false',
            'claims_opened_false',
            'final_controls_remain_failed',
            'dedicated_controls_status_recorded_from_artifacts',
            'all_required_dedicated_control_rows_audited',
            'blocking_controls_recorded',
            'threshold_source_review_recorded',
            'implementation_review_recorded',
        ],
        'observed_failed_dedicated_controls': sorted(observed_failed_dedicated),
        'observed_blocking_controls': sorted(observed_blocking_controls),
        'teacher_threshold_path_mismatch_suspected': summary.get('teacher_threshold_path_mismatch_suspected'),
        'implementation_review_blockers': implementation_review.get('blockers', []),
        'allowed_interpretation': 'Engineering controls remediation audit only; failed-control membership is read from generated artifacts.',
        'not_allowed_interpretation': 'Do not force old failed-control membership, treat this audit as evidence that controls passed, or open Phase 1 claims.',
        'not_ok_to_claim': claim_state.get('not_ok_to_claim', []),
    }
    write_json(latest_run / 'phase1_final_controls_remediation_audit_review_note.json', review_note)
    print('Review note written:', latest_run / 'phase1_final_controls_remediation_audit_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch flag is False. This is expected during first-pass manual hold.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Final Controls Remediation Audit Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_CONTROLS_REMEDIATION_AUDIT)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Final remediation plan run:', FINAL_REMEDIATION_PLAN_RUN)
print('Final controls run:', FINAL_CONTROLS_RUN)
print('Final dedicated controls run:', FINAL_DEDICATED_CONTROLS_RUN)

if not RUN_FINAL_CONTROLS_REMEDIATION_AUDIT:
    print('HELD: Controls remediation audit was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
else:
    print('Latest final controls-remediation-audit output:', latest_run)
    print('Control suite passed:', summary.get('control_suite_passed'))
    print('Dedicated control suite passed:', summary.get('dedicated_control_suite_passed'))
    print('Failed dedicated controls:', summary.get('failed_dedicated_controls'))
    print('Teacher threshold path mismatch suspected:', summary.get('teacher_threshold_path_mismatch_suspected'))
    print('CHECK REQUIRED: Review phase1_final_controls_remediation_decision_memo.md before any code/config remediation.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
